In [0]:
%sql  
select * from com_edp_prd.com_intgr.survey_target

In [0]:
%sql
select * from com_edp_prd.com_intgr.question_response

In [0]:

survey_target_df = spark.sql("""
SELECT a.*
FROM com_edp_prd.com_intgr.survey_target AS a
JOIN (
    SELECT 
        id,
        MAX(CAST(modified_date__v AS TIMESTAMP)) AS max_modified_date
    FROM com_edp_prd.com_intgr.survey_target
    GROUP BY id
) AS b 
    ON a.id = b.id 
    AND CAST(a.modified_date__v AS TIMESTAMP) = b.max_modified_date
""").toPandas()

question_response_df = spark.sql("""
                                 select a.*
from com_edp_prd.com_intgr.question_response as a
join (
  select survey_target__v, ctrl_survey__v, survey_question__v,  MAX(CAST(modified_date__v AS TIMESTAMP)) AS max_modified_date
from com_edp_prd.com_intgr.question_response
group by 1,2,3
) as b on a.survey_target__v = b.survey_target__v and a.ctrl_survey__v = b.ctrl_survey__v and a.survey_question__v = b.survey_question__v and cast(a.modified_date__v as timestamp) = b.max_modified_date
                                 """).toPandas()


In [0]:

print(survey_target_df.shape)
print(question_response_df.shape)

In [0]:

print(survey_target_df['id'].nunique())
print(question_response_df['survey_target__v'].nunique())

In [0]:

display(survey_target_df)

In [0]:

display(question_response_df)

In [0]:

print(survey_target_df.shape)
print(question_response_df.shape)

In [0]:

import pandas as pd

# Assuming your DataFrame is named question_response_df
# and has the columns: 'response__v', 'number__v', 'text__v'

question_response_df["final_response"] = question_response_df.apply(
    lambda row: next(
        (val for val in [row["response__v"], row["number__v"], row["text__v"]] if pd.notna(val)),
        None
    ),
    axis=1
)

question_response_df["final_response"] = question_response_df["final_response"].apply(
    lambda x: str(int(x)) if isinstance(x, float) and str(x).split('.')[-1] == '0' else str(x)
)

# question_response_df['final_response'] = question_response_df['final_response'].astype(str)

In [0]:

account_survey_responses = (
    question_response_df
    .merge(
        survey_target_df,
        left_on="survey_target__v",
        right_on="id",
        how="inner",
        suffixes=("_q", "_s")
    )[
        [
            "survey_target__v", #Account ID Joining Key
            "ctrl_survey__v", #Survey ID
            "order__v", #Order
            "survey_question__v", #Question ID
            "question_text__v", #Question Text
            "answer_choice__v", #Answer Choice
            "final_response", #Response
            "account__v", #Account ID
            "name__v_s", #Survey Type - HCP or HCO Survey
            "account_display_name__v" #Account Name
        ]
    ]
)


In [0]:


account_survey_responses = account_survey_responses.rename(columns={
    "survey_target__v" : "survey_target_id", #Will be different for same accounts
    "ctrl_survey__v": "survey_id",
    "order__v": "question_order",
    "survey_question__v": "question_id",
    "question_text__v": "question",
    "answer_choice__v": "response_type",
    "final_response": "response",
    "account__v": "account_id",
    "name__v_s": "survey_type",
    "account_display_name__v": "account_name"
})

In [0]:

display(account_survey_responses)

In [0]:


account_survey_responses = (
    account_survey_responses
    .sort_values(by=['survey_target_id', 'survey_id', 'question_order'], ascending=True)
    .reset_index(drop=True)
)

In [0]:


account_survey_responses['question_order'] += 1

In [0]:

display(account_survey_responses)

In [0]:


account_survey_responses['response_type'].isna().sum()

In [0]:



mapping = {
    "Yes;0;No;0": "Yes/No",
    "Home Infusion;0;Hospital;0;HOPD;0;AIC;0;Other;0": "Home Infusion/Hospital/Hospital Outpatient Department (HOPD)/Ambulatory Infusion Center (AIC)/Other",
    "SP;0;SD;0;HOPD;0;Other;0": "SP (Specialty Pharmacy)/SD (Specialty Distributor)/HOPD/Other",
    "Very;0;Somewhat;0;Not at all;0": "Very/Somewhat/Not at all",
    # "null": "Subjective Question"
}

account_survey_responses["response_type"] = (
    account_survey_responses["response_type"]
    .replace(mapping)
    .fillna("Subjective Question")
)

print(account_survey_responses[account_survey_responses['response_type'] == 'Subjective Question'].shape)

display(account_survey_responses)


In [0]:


print(account_survey_responses['response'].isna().sum())
account_survey_responses['response'] = account_survey_responses['response'].fillna('')
print(account_survey_responses['response'].isna().sum())

### Fetching Account info from crm tables

In [0]:

display(account_survey_responses.head(5))

In [0]:

unique_combinations = account_survey_responses[['survey_type', 'survey_target_id', 'account_id', 'account_name']].drop_duplicates()
display(unique_combinations)

In [0]:
%sql
SELECT DISTINCT
  a.npi__v AS npi,
  a.id AS account_id,
  a.name__v AS name,
  CASE 
    WHEN a.postal_code_cda__v IS NULL THEN NULL
    WHEN POSITION('-' IN a.postal_code_cda__v) > 0 THEN SUBSTRING(a.postal_code_cda__v, 1, POSITION('-' IN a.postal_code_cda__v) - 1)
    ELSE a.postal_code_cda__v
  END AS postal_code,

  b.npi__v AS parent_npi,
  a.primary_parent__v AS parent_account_id,
  b.name__v AS parent_name,
  CASE 
    WHEN b.postal_code_cda__v IS NULL THEN NULL
    WHEN POSITION('-' IN b.postal_code_cda__v) > 0 THEN SUBSTRING(b.postal_code_cda__v, 1, POSITION('-' IN b.postal_code_cda__v) - 1)
    ELSE b.postal_code_cda__v
  END AS parent_postal_code

FROM com_edp_prd.com_raw.vcrm_account__v AS a
LEFT JOIN com_edp_prd.com_raw.vcrm_account__v AS b
  ON a.primary_parent__v = b.id
ORDER BY account_id;


In [0]:

vcrm_account_df = spark.sql("""
SELECT DISTINCT
  a.npi__v AS npi,
  a.id AS account_id,
  a.name__v AS name,
  CASE 
    WHEN a.postal_code_cda__v IS NULL THEN NULL
    WHEN POSITION('-' IN a.postal_code_cda__v) > 0 THEN SUBSTRING(a.postal_code_cda__v, 1, POSITION('-' IN a.postal_code_cda__v) - 1)
    ELSE a.postal_code_cda__v
  END AS postal_code,

  b.npi__v AS parent_npi,
  a.primary_parent__v AS parent_account_id,
  b.name__v AS parent_name,
  CASE 
    WHEN b.postal_code_cda__v IS NULL THEN NULL
    WHEN POSITION('-' IN b.postal_code_cda__v) > 0 THEN SUBSTRING(b.postal_code_cda__v, 1, POSITION('-' IN b.postal_code_cda__v) - 1)
    ELSE b.postal_code_cda__v
  END AS parent_postal_code

FROM com_edp_prd.com_raw.vcrm_account__v AS a
LEFT JOIN com_edp_prd.com_raw.vcrm_account__v AS b
  ON a.primary_parent__v = b.id
ORDER BY account_id
""").toPandas()


In [0]:

vcrm_account_df.display()

In [0]:

merged_df = unique_combinations.merge(
    vcrm_account_df,
    on='account_id',
    how='left'
    )
display(merged_df)

In [0]:

target_hcp_hco_mapping = spark.sql("""
                                   select * from com_edp_prd.cmpa_insights_internal_schema.target_hcp_hco_mapping
                                   """).toPandas()

In [0]:

target_hcp_hco_mapping.columns

In [0]:

merged_df.columns

In [0]:

target_hcp_hco_mapping['hcp_npi'].nunique()

In [0]:

# assume pandas is imported and these DataFrames already exist:
# merged_df, target_hcp_hco_mapping
import pandas as pd

# 1) prepare the 'b' table (target_hcp_hco_mapping) selection + derived hcp_name_2
b_sel = target_hcp_hco_mapping[[
    "hcp_npi",
    "hcp_first_name",
    "hcp_last_name",
    "hcp_zipcode",
    "hco_primary_npi",
    "hco_primary_name",
    "hco_zipcode"
]].copy()

# create concatenated name (mimics concat_ws(' ', first, last))
b_sel["hcp_name_2"] = (b_sel["hcp_first_name"].fillna("") + " " + b_sel["hcp_last_name"].fillna("")).str.strip()

# pick and rename the columns we want from b for merging
b_for_merge = b_sel[[
    "hcp_npi",
    "hcp_name_2",
    "hcp_zipcode",
    "hco_primary_npi",
    "hco_primary_name",
    "hco_zipcode"
]].rename(columns={
    "hcp_zipcode": "hcp_postal_code_2",
    "hco_primary_npi": "hcp_primary_hco_npi_2",
    "hco_primary_name": "hcp_hco_primary_name_2",
    "hco_zipcode": "hcp_hco_zipcode_2"
})

# 2) left join merged_df (a) with b_for_merge on npi == hcp_npi
df1 = merged_df.merge(
    b_for_merge,
    left_on="npi",
    right_on="hcp_npi",
    how="left"
).drop(columns=["hcp_npi"])  # drop helper join key if not needed

# 3) prepare the distinct 'c' table (distinct hco_primary_npi, name, zipcode)
c_sel = target_hcp_hco_mapping[[
    "hco_primary_npi",
    "hco_primary_name",
    "hco_zipcode"
]].drop_duplicates(subset=["hco_primary_npi"]).rename(columns={
    "hco_primary_name": "hco_primary_name_3",
    "hco_zipcode": "hco_zipcode_3"
})

# 4) left join df1 with c_sel on parent_npi == hco_primary_npi
final_df = df1.merge(
    c_sel,
    left_on="parent_npi",
    right_on="hco_primary_npi",
    how="left"
).drop(columns=["hco_primary_npi"])  # drop helper key

# final_df now corresponds to the SQL SELECT a.*, ... with the renamed fields
display(final_df)

In [0]:

final_df.columns

In [0]:

import pandas as pd
import numpy as np

# assume final_df already exists

# helper: safely pick first non-null between two columns
def first_non_null(a, b):
    # returns a Series where for each row the first non-null of a then b is chosen
    return a.combine_first(b)

# 1) postal_code_final
# - For Denali HCO Survey: prefer postal_code then hco_zipcode_3
# - For Denali HCP Survey: prefer postal_code then hcp_postal_code_2
# - Else: NA
is_hco = final_df["survey_type"] == "Denali HCO Survey"
is_hcp = final_df["survey_type"] == "Denali HCP Survey"

# compute candidates
postal_candidate_hco = first_non_null(final_df["postal_code"], final_df["hco_zipcode_3"])
postal_candidate_hcp = first_non_null(final_df["postal_code"], final_df["hcp_postal_code_2"])

final_df["postal_code_final"] = pd.NA
final_df.loc[is_hco, "postal_code_final"] = postal_candidate_hco.loc[is_hco]
final_df.loc[is_hcp, "postal_code_final"] = postal_candidate_hcp.loc[is_hcp]

# 2) parent_npi_final
# - Only for Denali HCP Survey: prefer parent_npi then hcp_primary_hco_npi_2
final_df["parent_npi_final"] = pd.NA
parent_npi_candidate = first_non_null(final_df["parent_npi"], final_df["hcp_primary_hco_npi_2"])
final_df.loc[is_hcp, "parent_npi_final"] = parent_npi_candidate.loc[is_hcp]

# 3) parent_name_final
# - Only for Denali HCP Survey: prefer parent_name then hcp_hco_primary_name_2
final_df["parent_name_final"] = pd.NA
parent_name_candidate = first_non_null(final_df["parent_name"], final_df["hcp_hco_primary_name_2"])
final_df.loc[is_hcp, "parent_name_final"] = parent_name_candidate.loc[is_hcp]

# 4) parent_zip_final
# - Only for Denali HCP Survey: prefer parent_postal_code then hcp_hco_zipcode_2
final_df["parent_zip_final"] = pd.NA
parent_zip_candidate = first_non_null(final_df["parent_postal_code"], final_df["hcp_hco_zipcode_2"])
final_df.loc[is_hcp, "parent_zip_final"] = parent_zip_candidate.loc[is_hcp]

# OPTIONAL: if you want to normalize empty strings to NA
final_df.replace({"": pd.NA}, inplace=True)

# Build mapping_df with requested columns (note the corrected closing quote)
mapping_df = final_df[[
    "survey_type",
    "survey_target_id",
    "account_id",
    "account_name",
    "npi",
    "postal_code_final",
    "parent_npi_final",
    "parent_name_final",
    "parent_zip_final"
]].copy()

# (Optional) reset index if you prefer
mapping_df.reset_index(drop=True, inplace=True)


In [0]:

display(mapping_df)

### For getting the max modified date

In [0]:
%sql
create or replace temporary view target as 
SELECT a.*
FROM com_edp_prd.com_intgr.survey_target AS a
JOIN (
    SELECT 
        id,
        MAX(CAST(modified_date__v AS TIMESTAMP)) AS max_modified_date
    FROM com_edp_prd.com_intgr.survey_target
    GROUP BY id
) AS b 
    ON a.id = b.id 
    AND CAST(a.modified_date__v AS TIMESTAMP) = b.max_modified_date

In [0]:
%sql
create or replace temporary view response as 
select a.*
from com_edp_prd.com_intgr.question_response as a
join (
  select survey_target__v, ctrl_survey__v, survey_question__v,  MAX(CAST(modified_date__v AS TIMESTAMP)) AS max_modified_date
from com_edp_prd.com_intgr.question_response
group by 1,2,3
) as b on a.survey_target__v = b.survey_target__v and a.ctrl_survey__v = b.ctrl_survey__v and a.survey_question__v = b.survey_question__v and cast(a.modified_date__v as timestamp) = b.max_modified_date;

In [0]:
%sql
select a.name__v, b.survey_target__v, a.account_display_name__v, MAX(CAST(b.modified_date__v AS date)) AS max_modified_date
from target as a join response as b on a.id = b.survey_target__v
group by 1,2,3 order by 1,2,3

In [0]:

date_df = spark.sql("""
                                   select a.name__v, b.survey_target__v, a.account_display_name__v, MAX(CAST(b.modified_date__v AS date)) AS max_modified_date
from target as a join response as b on a.id = b.survey_target__v
group by 1,2,3 order by 1,2,3
                                   """).toPandas()

In [0]:

date_df.columns

In [0]:

# merge mapping_df with date_df on survey_target_id ↔ survey_target__v
mapping_df = mapping_df.merge(
    date_df[["survey_target__v", "max_modified_date"]],
    how="left",
    left_on="survey_target_id",
    right_on="survey_target__v"
)

# optional: drop the redundant join key from date_df
mapping_df.drop(columns=["survey_target__v"], inplace=True)


In [0]:

display(mapping_df)

### Mapping ZIP

In [0]:

zip_to_tier_df = spark.sql("select distinct a.*, b.account_lead, b.region_name, b.rbd from com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping as a left join com_edp_prd.cmpa_insights_internal_schema.territory_al_rbd as b on a.territory_name = b.territory_name").toPandas()

In [0]:

zip_to_tier_df.columns

In [0]:

# --- Ensure all ZIP columns are strings before merging ---
mapping_df["postal_code_final"] = mapping_df["postal_code_final"].astype(str)
mapping_df["parent_zip_final"]  = mapping_df["parent_zip_final"].astype(str)
zip_to_tier_df["zipcode"]       = zip_to_tier_df["zipcode"].astype(str)

# 1️⃣ First join: mapping_df (a) ←→ zip_to_tier_df (b)
merged_df = mapping_df.merge(
    zip_to_tier_df[["zipcode", "territory_name", "account_lead"]],
    how="left",
    left_on="postal_code_final",
    right_on="zipcode"
)

# 3️⃣ Second join: merged_df (a) ←→ zip_to_tier_df (c)
merged_df = merged_df.merge(
    zip_to_tier_df[["zipcode", "territory_name"]],
    how="left",
    left_on="parent_zip_final",
    right_on="zipcode",
    suffixes=("", "_parent")
)

# 4️⃣ Rename parent columns properly
merged_df.rename(
    columns={"territory_name_parent": "parent_territory"},
    inplace=True
)

# 5️⃣ Drop redundant columns
merged_df.drop(columns=["zipcode", "zipcode_parent"], errors="ignore", inplace=True)

# Final output
final_output = merged_df


In [0]:

display(final_output)

In [0]:
# Step 1: get max max_modified_date per account_id
t1 = (
    final_output
    .groupby("account_id", as_index=False)["max_modified_date"]
    .max()
    .rename(columns={"max_modified_date": "max_date"})
)

# Step 2: join back to get full rows
t2 = final_output.merge(
    t1,
    left_on=["account_id", "max_modified_date"],
    right_on=["account_id", "max_date"],
    how="inner"
)

# Optional: drop helper column
final_output_v2 = t2.drop(columns=["max_date"])


In [0]:
display(final_output_v2)

In [0]:
account_survey_responses = account_survey_responses[account_survey_responses['survey_target_id'].isin(final_output_v2['survey_target_id'].tolist())]

In [0]:
hcp_df = account_survey_responses[account_survey_responses['survey_type'] == 'Denali HCP Survey']
hco_df = account_survey_responses[account_survey_responses['survey_type'] == 'Denali HCO Survey']
msl_df = account_survey_responses[account_survey_responses['survey_type'] == '2025 MPS II Treatment Landscape Input Plan']

### HCP

In [0]:
hcp_df.columns


In [0]:

import pandas as pd

# Example function to detect ALL CAPS (ignoring numbers & punctuation)
def is_all_caps(text):
    return isinstance(text, str) and text.isupper()

# Create a new column 'section'
hcp_df['section'] = hcp_df['question'].where(
    hcp_df['question'].apply(is_all_caps)
)

# Forward-fill the last seen ALL CAPS value down the column
hcp_df['section'] = hcp_df['section'].ffill()

In [0]:


# Filter out rows where 'question' is ALL CAPS
hcp_df = hcp_df[
    ~hcp_df['question'].apply(lambda x: isinstance(x, str) and x.isupper())
].reset_index(drop=True)

In [0]:

# Step 1: Sort the dataframe
hcp_df = hcp_df.sort_values(
    by=['survey_target_id', 'survey_id', 'question_order'],
    ascending=[True, True, True]
).reset_index(drop=True)

# Step 2: Create new_question_order (partitioned by account_id)
hcp_df['new_question_order'] = (
    hcp_df.groupby('survey_target_id').cumcount() + 1
)

# Step 3: Drop old question_order column
hcp_df = hcp_df.drop(columns=['question_order'])


In [0]:

hcp_df.display()

In [0]:


# create pivot with account_name and survey_target_id as separate column levels
hcp_pivot = hcp_df.pivot_table(
    index=['section', 'new_question_order', 'question'],
    columns=['account_name', 'survey_target_id'],
    values='response',
    aggfunc='first'
).reset_index()

# flatten the MultiIndex columns into single-level column names like "accountName__surveyTargetId"
hcp_pivot.columns = [
    col if not isinstance(col, tuple) else f"{col[0]}__{col[1]}"
    for col in hcp_pivot.columns
]

In [0]:
display(hcp_pivot)

In [0]:

hcp_pivot = hcp_pivot.sort_values(by=['new_question_order__'], ascending=[True])

In [0]:
hcp_pivot.columns = [
    col.split('__')[1] if '__' in col else col
    for col in hcp_pivot.columns
]

In [0]:

hcp_pivot.display()

### HCO

In [0]:
hco_df.columns


In [0]:

import pandas as pd

# Example function to detect ALL CAPS (ignoring numbers & punctuation)
def is_all_caps(text):
    return isinstance(text, str) and text.isupper()

# Create a new column 'section'
hco_df['section'] = hco_df['question'].where(
    hco_df['question'].apply(is_all_caps)
)

# Forward-fill the last seen ALL CAPS value down the column
hco_df['section'] = hco_df['section'].ffill()

In [0]:


# Filter out rows where 'question' is ALL CAPS
hco_df = hco_df[
    ~hco_df['question'].apply(lambda x: isinstance(x, str) and x.isupper())
].reset_index(drop=True)

In [0]:

# Step 1: Sort the dataframe
hco_df = hco_df.sort_values(
    by=['survey_target_id', 'survey_id', 'question_order'],
    ascending=[True, True, True]
).reset_index(drop=True)

# Step 2: Create new_question_order (partitioned by account_id)
hco_df['new_question_order'] = (
    hco_df.groupby('survey_target_id').cumcount() + 1
)

# Step 3: Drop old question_order column
hco_df = hco_df.drop(columns=['question_order'])


In [0]:

hco_df.display()

In [0]:

hco_df.columns

In [0]:


# create pivot with account_name and survey_target_id as separate column levels
hco_pivot = hco_df.pivot_table(
    index=['section', 'new_question_order', 'question'],
    columns=['account_name', 'survey_target_id'],
    values='response',
    aggfunc='first'
).reset_index()

# flatten the MultiIndex columns into single-level column names like "accountName__surveyTargetId"
hco_pivot.columns = [
    col if not isinstance(col, tuple) else f"{col[0]}__{col[1]}"
    for col in hco_pivot.columns
]

In [0]:
display(hco_pivot)

In [0]:

hco_pivot = hco_pivot.sort_values(by=['new_question_order__'], ascending=[True])

In [0]:
hco_pivot.columns = [
    col.split('__')[1] if '__' in col else col
    for col in hco_pivot.columns
]

In [0]:

hco_pivot.display()

### msl survey

In [0]:

msl_df.columns


In [0]:

import pandas as pd

# Example function to detect ALL CAPS (ignoring numbers & punctuation)
def is_all_caps(text):
    return isinstance(text, str) and text.isupper()

# Create a new column 'section'
msl_df['section'] = msl_df['question'].where(
    msl_df['question'].apply(is_all_caps)
)

# Forward-fill the last seen ALL CAPS value down the column
msl_df['section'] = msl_df['section'].ffill()

In [0]:




# Filter out rows where 'question' is ALL CAPS
msl_df = msl_df[
    ~msl_df['question'].apply(lambda x: isinstance(x, str) and x.isupper())
].reset_index(drop=True)

In [0]:


# Step 1: Sort the dataframe
msl_df = msl_df.sort_values(
    by=['survey_target_id', 'survey_id', 'question_order'],
    ascending=[True, True, True]
).reset_index(drop=True)

# Step 2: Create new_question_order (partitioned by account_id)
msl_df['new_question_order'] = (
    msl_df.groupby('account_id').cumcount() + 1
)

# Step 3: Drop old question_order column
msl_df = msl_df.drop(columns=['question_order'])


In [0]:

msl_df.display()

In [0]:

msl_df.columns

In [0]:

msl_pivot = msl_df.pivot_table(
    index=['section', 'new_question_order', 'question'],
    columns='survey_target_id',
    values='response',
    aggfunc='first'   # in case there's only one response per (account_id, question_id)
).reset_index()

In [0]:
msl_pivot = msl_pivot.sort_values(by=['new_question_order'], ascending=[True])

In [0]:
msl_pivot.display()

## Getting patient count numbers

In [0]:
%sql
with t1 as (
  select * from final_output
),
hcp_patient_numbers as (
  select distinct survey_target_id, response as patient_count
  from hcp_df
  where question_id = 'VC2000000001002'
),
hco_patient_numbers as (
  select distinct survey_target_id, response as patient_count
  from hco_df 
  where question_id = 'VC2000000001024'
)

In [0]:
final_output.columns

In [0]:
import numpy as np
import pandas as pd

# t1 equivalent
t1 = final_output_v2.copy()

# hcp_patient_numbers equivalent
hcp_patient_numbers = (
    hcp_df.loc[hcp_df["question_id"] == "VC2000000001002",
               ["survey_target_id", "response"]]
    .drop_duplicates()
    .rename(columns={"response": "patient_count"})
)

# hco_patient_numbers equivalent
hco_patient_numbers = (
    hco_df.loc[hco_df["question_id"] == "VC2000000001024",
               ["survey_target_id", "response"]]
    .drop_duplicates()
    .rename(columns={"response": "patient_count"})
)

# Left join both patient tables
df = (
    t1
    .merge(
        hcp_patient_numbers,
        on="survey_target_id",
        how="left",
        suffixes=("", "_hcp")
    )
    .merge(
        hco_patient_numbers,
        on="survey_target_id",
        how="left",
        suffixes=("", "_hco")
    )
)

# Conditional patient_count logic (CASE WHEN equivalent)
df["patient_count"] = np.where(
    df["survey_type"] == "Denali HCP Survey",
    df["patient_count"],
    np.where(
        df["survey_type"] == "Denali HCO Survey",
        df["patient_count_hco"],
        np.nan
    )
)

# Drop intermediate columns
df = df.drop(columns=["patient_count_hco"])

# df is the final output



In [0]:
display(df)

In [0]:
df = df[(df['survey_type'] != '2025 MPS II Treatment Landscape Input Plan') & (~(df['patient_count'].isna())) & (df['patient_count'] != 'None')].display()

In [0]:
%sql
-- with unique_hcos as ()
with patient_count_df as (
  select * from df where account_id not in ('V4T000000001182', 'V4T000000011039')
),
unique_hcos as (
  select distinct parent_npi_final as hco_npi, parent_name_final as hco_name
from patient_count_df
where parent_npi_final is not null and survey_type = 'Denali HCP Survey'
union 
select distinct npi as hco_npi, account_name as hco_name
from patient_count_df
where npi is not null and survey_type = 'Denali HCO Survey'
),
hcp_level_rollup as (
  select a.parent_npi_final as hco_npi, sum(a.patient_count) as patient_counts_hcp_aggregated
  from patient_count_df as a
  where a.survey_type = 'Denali HCP Survey' 
  group by 1
),
hco_level_rollup as (
  select distinct a.npi, a.patient_count
  from patient_count_df 
  where a.survey_type = 'Denali HCO Survey'
)
select a.*
from unique_hcos as a
left join hcp_level_rollup as b on a.hco_npi = b.

In [0]:
import pandas as pd

# --------------------------------------------------
# Step 0: Filter excluded account_ids
# --------------------------------------------------
patient_count_df = df[
    ~df["account_id"].isin(["V4T000000001182", "V4T000000011039"])
].copy()

# --------------------------------------------------
# Step 1: HCO sources (UNION ALL equivalent)
# --------------------------------------------------
hcp_hco_sources = patient_count_df.loc[
    (patient_count_df["survey_type"] == "Denali HCP Survey") &
    (patient_count_df["parent_npi_final"].notna()),
    ["parent_npi_final", "parent_name_final"]
].rename(columns={
    "parent_npi_final": "hco_npi",
    "parent_name_final": "hco_name"
})

hco_hco_sources = patient_count_df.loc[
    (patient_count_df["survey_type"] == "Denali HCO Survey") &
    (patient_count_df["npi"].notna()),
    ["npi", "account_name"]
].rename(columns={
    "npi": "hco_npi",
    "account_name": "hco_name"
})

hco_sources = pd.concat(
    [hcp_hco_sources, hco_hco_sources],
    ignore_index=True
)

# --------------------------------------------------
# Step 2: Explicit HCO de-duplication
# --------------------------------------------------
unique_hcos = (
    hco_sources
    .groupby("hco_npi", as_index=False)
    .agg({"hco_name": "max"})
)

# --------------------------------------------------
# Step 3: HCP → HCO rollup
# --------------------------------------------------
hcp_level_rollup = (
    patient_count_df
    .loc[patient_count_df["survey_type"] == "Denali HCP Survey"]
    .groupby("parent_npi_final", as_index=False)
    .agg(patient_counts_hcp_aggregated=("patient_count", "sum"))
    .rename(columns={"parent_npi_final": "hco_npi"})
)

# --------------------------------------------------
# Step 4: HCO-level rollup (deterministic)
# --------------------------------------------------
hco_level_rollup = (
    patient_count_df
    .loc[patient_count_df["survey_type"] == "Denali HCO Survey"]
    .groupby("npi", as_index=False)
    .agg(patient_count_hco_level=("patient_count", "max"))
    .rename(columns={"npi": "hco_npi"})
)

# --------------------------------------------------
# Step 5: Final joins
# --------------------------------------------------
final_df = (
    unique_hcos
    .merge(hcp_level_rollup, on="hco_npi", how="left")
    .merge(hco_level_rollup, on="hco_npi", how="left")
)
